In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from geopy.distance import geodesic
import math
import asyncio
from playwright.async_api import async_playwright
import nest_asyncio
nest_asyncio.apply()

## SCRAPPING COIN AFRIQUE

In [8]:
async def scraper_coinafrique_robuste(nb_pages=10):
  annonces = []

  async with async_playwright() as p:
    # Lancement du navigateur
    browser = await p.chromium.launch(headless=True)
    page = await browser.new_page()

    for page_num in range(1, nb_pages + 1):
      url = f"https://sn.coinafrique.com/categorie/appartements?page={page_num}"
      print(f"🚀 Chargement de la page {page_num}/{nb_pages}...")

      try:
        await page.goto(
            url, wait_until="domcontentloaded", timeout=60000
        )

        # SCROLL AUTOMATIQUE : Oblige le site à charger TOUTES les annonces de la page
        for _ in range(3):
          await page.evaluate("window.scrollBy(0, 1000);")
          await page.wait_for_timeout(800)

        content = await page.content()
        soup = BeautifulSoup(content, "html.parser")

        # Sélecteur plus large pour attraper toutes les cartes d'annonces
        cards = soup.find_all("div", class_="card-content")

        count_page = 0
        for card in cards:
          try:
            # Titre
            titre_elem = card.find("p", class_="ad__card-description")
            titre = titre_elem.text.strip() if titre_elem else "N/A"

            # Prix
            prix_elem = card.find("p", class_="ad__card-price")
            prix_raw = prix_elem.text.strip() if prix_elem else "0"
            prix_clean = re.sub(r"[^\d]", "", prix_raw)
            prix = int(prix_clean) if prix_clean else None

            # Adresse & Quartier
            loc_elem = card.find("p", class_="ad__card-location")
            loc_raw = loc_elem.text.strip() if loc_elem else "Dakar"
            adresse_clean = loc_raw.replace("location_on", "").strip()
            quartier = adresse_clean.split(",")[0].strip()

            # Lien
            parent_a = card.find_parent("a")
            lien = (
                "https://sn.coinafrique.com" + parent_a["href"]
                if parent_a and "href" in parent_a.attrs
                else "N/A"
            )

            # Image
            card_container = card.find_parent("div", class_="card")
            img_elem = card_container.find("img") if card_container else None
            img_url = "N/A"
            if img_elem:
              img_url = (
                  img_elem.get("src") or img_elem.get("data-src") or "N/A"
              )

            if prix and prix > 10000:
              annonces.append({
                  "titre": titre,
                  "prix_affiche_fcfa": prix,
                  "quartier": quartier,
                  "adresse_complete": adresse_clean,
                  "image_url": img_url,
                  "url_annonce": lien,
                  "source": "CoinAfrique",
              })
              count_page += 1
          except Exception:
            continue

        print(f"  └ {count_page} annonces récupérées sur la page {page_num}")

      except Exception as e:
        print(f"⚠️ Erreur sur la page {page_num}: {e}")
        continue

    await browser.close()

  # Suppression des doublons stricts (si une même annonce est récupérée 2 fois)
  df = pd.DataFrame(annonces)
  df = df.drop_duplicates(subset=["titre", "prix_affiche_fcfa", "quartier"])
  return df


# Exécution dans Jupyter
df_coinafrique = await scraper_coinafrique_robuste(nb_pages=10)

# Exportation Excel
df_coinafrique.to_csv(
    "dataset_coinafrique_10pages.csv", index=False, sep=";", encoding="utf-8-sig"
)
print(
    f"\n✅ Total d'annonces uniques extraites : {len(df_coinafrique)} annonces !"
)

🚀 Chargement de la page 1/10...
  └ 84 annonces récupérées sur la page 1
🚀 Chargement de la page 2/10...
  └ 84 annonces récupérées sur la page 2
🚀 Chargement de la page 3/10...
  └ 84 annonces récupérées sur la page 3
🚀 Chargement de la page 4/10...
  └ 83 annonces récupérées sur la page 4
🚀 Chargement de la page 5/10...
  └ 79 annonces récupérées sur la page 5
🚀 Chargement de la page 6/10...
  └ 83 annonces récupérées sur la page 6
🚀 Chargement de la page 7/10...
  └ 81 annonces récupérées sur la page 7
🚀 Chargement de la page 8/10...
  └ 80 annonces récupérées sur la page 8
🚀 Chargement de la page 9/10...
  └ 83 annonces récupérées sur la page 9
🚀 Chargement de la page 10/10...
  └ 83 annonces récupérées sur la page 10

✅ Total d'annonces uniques extraites : 793 annonces !


## SCRAPPING FACEBOOK MARKETPLACE

In [13]:
nest_asyncio.apply()

async def scraper_facebook_avec_session(target_count=500):
    url = "https://www.facebook.com/marketplace/106355749403409/propertyrentals"
    annonces = []
    seen_urls = set()

    print(f"🚀 Lancement de Facebook Marketplace (Objectif : {target_count} annonces)...")

    async with async_playwright() as p:
        # Utilisation d'un dossier de session pour sauvegarder la connexion Facebook
        # 'user_data_dir' enregistre vos cookies sur votre machine
        context = await p.chromium.launch_persistent_context(
            user_data_dir="./fb_session_data",
            headless=False, # Fenêtre visible obligatoire pour se connecter
            viewport={"width": 1280, "height": 800},
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        
        page = context.pages[0] if context.pages else await context.new_page()
        await page.goto(url, wait_until="domcontentloaded")
        await asyncio.sleep(3)

        # Vérification si la fenêtre de connexion apparaît
        print("\n🔑 SI UNE FENÊTRE DE CONNEXION FACEBOOK S'AFFICHE :")
        print(" Connectez-vous à votre compte Facebook dans le navigateur ouvert.")
        input("👉 Une fois connecté et sur la page du Marketplace, appuyez sur ENTRÉE ici dans le terminal pour lancer le scraping...")

        print("\n🚀 Début du scraping avec défilement infini...")

        consecutive_no_new = 0

        while len(seen_urls) < target_count:
            # Récupération des cartes d'annonces
            cards = await page.locator('a[href*="/marketplace/item/"]').all()
            initial_count = len(seen_urls)

            for card in cards:
                try:
                    href = await card.get_attribute('href')
                    if not href:
                        continue
                    
                    clean_url = "https://www.facebook.com" + href.split('?')[0] if href.startswith('/') else href.split('?')[0]

                    if clean_url not in seen_urls:
                        url_annonce = clean_url
                        seen_urls.add(clean_url)

                        # Extraction du texte (Prix, Titre, Localisation)
                        card_text = await card.inner_text()
                        lines = [line.strip() for line in card_text.split('\n') if line.strip()]

                        prix_raw = lines[0] if len(lines) > 0 else "0"
                        prix_clean = re.sub(r'[^\d]', '', prix_raw)
                        prix = int(prix_clean) if prix_clean else None

                        titre = lines[1] if len(lines) > 1 else "N/A"
                        adresse_complete = lines[2] if len(lines) > 2 else "Dakar, Sénégal"
                        quartier = adresse_complete.split(',')[0].strip()

                        # Extraction de l'image
                        img_elem = card.locator('img')
                        img_url = "N/A"
                        if await img_elem.count() > 0:
                            img_url = await img_elem.first.get_attribute('src') or "N/A"

                        if prix and prix > 10000:
                            annonces.append({
                                'titre': titre,
                                'prix_affiche_fcfa': prix,
                                'quartier': quartier,
                                'adresse_complete': adresse_complete,
                                'image_url': img_url,
                                'url_annonce': url_annonce,
                                'source': 'Facebook Marketplace'
                            })

                        if len(seen_urls) >= target_count:
                            break
                except Exception:
                    continue

            print(f"  └ Annonces uniques collectées : {len(seen_urls)} / {target_count}")

            if len(seen_urls) == initial_count:
                consecutive_no_new += 1
                if consecutive_no_new >= 6:
                    print("⚠️ Fin du flux d'annonces ou chargement au bas de page atteint.")
                    break
            else:
                consecutive_no_new = 0

            # Scroll vers le bas pour charger le flux dynamique
            await page.evaluate("window.scrollBy(0, 1500);")
            await asyncio.sleep(2.5) # Temps suffisant pour que Facebook charge la suite

        await context.close()

    df = pd.DataFrame(annonces)
    return df

# Exécution dans Jupyter
df_facebook = await scraper_facebook_avec_session(target_count=500)

# Sauvegarde sous un nom indépendant
df_facebook.to_csv("dataset_facebook_500.csv", index=False, sep=';', encoding='utf-8-sig')
print(f"\n✅ Terminé ! Total extrais sur Facebook : {len(df_facebook)} annonces.")

🚀 Lancement de Facebook Marketplace (Objectif : 500 annonces)...

🔑 SI UNE FENÊTRE DE CONNEXION FACEBOOK S'AFFICHE :
 Connectez-vous à votre compte Facebook dans le navigateur ouvert.

🚀 Début du scraping avec défilement infini...
  └ Annonces uniques collectées : 24 / 500
  └ Annonces uniques collectées : 34 / 500
  └ Annonces uniques collectées : 44 / 500
  └ Annonces uniques collectées : 54 / 500


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/traceback.py:425: RuntimeWarning: coroutine 'scraper_coinafrique_playwright' was never awaited
  result.append(FrameSummary(


  └ Annonces uniques collectées : 64 / 500
  └ Annonces uniques collectées : 84 / 500
  └ Annonces uniques collectées : 94 / 500
  └ Annonces uniques collectées : 104 / 500
  └ Annonces uniques collectées : 114 / 500
  └ Annonces uniques collectées : 124 / 500
  └ Annonces uniques collectées : 144 / 500
  └ Annonces uniques collectées : 154 / 500
  └ Annonces uniques collectées : 164 / 500
  └ Annonces uniques collectées : 174 / 500
  └ Annonces uniques collectées : 192 / 500
  └ Annonces uniques collectées : 204 / 500
  └ Annonces uniques collectées : 214 / 500
  └ Annonces uniques collectées : 224 / 500
  └ Annonces uniques collectées : 234 / 500
  └ Annonces uniques collectées : 254 / 500
  └ Annonces uniques collectées : 263 / 500
  └ Annonces uniques collectées : 272 / 500
  └ Annonces uniques collectées : 282 / 500
  └ Annonces uniques collectées : 302 / 500
  └ Annonces uniques collectées : 312 / 500
  └ Annonces uniques collectées : 322 / 500
  └ Annonces uniques collectées : 3

## SCRAPPING EXPAT DAKAR

In [16]:
nest_asyncio.apply()

async def scraper_expat_dakar_robuste(nb_pages=50):
    annonces = []
    
    print(f"🚀 Début du scraping Expat-Dakar sur {nb_pages} pages...")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={"width": 1280, "height": 800}
        )
        page = await context.new_page()

        for page_num in range(1, nb_pages + 1):
            url = f"https://www.expat-dakar.com/appartements-a-louer?page={page_num}"
            print(f"🚀 Chargement de la page {page_num}/{nb_pages}...")

            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=60000)

                # Scroll automatique pour déclencher le chargement des images (Lazy Loading)
                for _ in range(3):
                    await page.evaluate("window.scrollBy(0, 1200);")
                    await page.wait_for_timeout(600)

                content = await page.content()
                soup = BeautifulSoup(content, 'html.parser')

                # Ciblage des cartes d'annonces sur Expat-Dakar
                listings = soup.find_all('div', class_=re.compile(r'listing-card'))

                count_page = 0
                for item in listings:
                    try:
                        # 1. Lien de l'annonce
                        a_tag = item.find('a', class_=re.compile(r'listing-card__inner'))
                        if not a_tag or 'href' not in a_tag.attrs:
                            continue
                        lien = a_tag['href']

                        # 2. Titre
                        titre_elem = item.find('div', class_=re.compile(r'listing-card__header__title'))
                        titre = titre_elem.text.strip() if titre_elem else "N/A"

                        # 3. Prix
                        prix_elem = item.find('span', class_=re.compile(r'listing-card__price__value'))
                        prix_raw = prix_elem.text.strip() if prix_elem else "0"
                        prix_clean = re.sub(r'[^\d]', '', prix_raw)
                        prix = int(prix_clean) if prix_clean else None

                        # 4. Localisation & Quartier
                        loc_elem = item.find('div', class_=re.compile(r'listing-card__header__location'))
                        adresse = loc_elem.text.strip() if loc_elem else "Dakar, Sénégal"
                        quartier = adresse.split(',')[0].replace('Dakar', '').strip()

                        # 5. URL de l'image
                        img_elem = item.find('img', class_=re.compile(r'listing-card__image__resource'))
                        img_url = "N/A"
                        if img_elem:
                            img_url = img_elem.get('src') or img_elem.get('data-src') or "N/A"

                        # Filtrage sur la fourchette de loyer valide
                        if prix and 20000 <= prix <= 5000000:
                            annonces.append({
                                'titre': titre,
                                'prix_affiche_fcfa': prix,
                                'quartier': quartier,
                                'adresse_complete': adresse,
                                'image_url': img_url,
                                'url_annonce': lien,
                                'source': 'Expat-Dakar'
                            })
                            count_page += 1
                    except Exception:
                        continue

                print(f"  └ {count_page} annonces récupérées sur la page {page_num}")

            except Exception as e:
                print(f"⚠️ Erreur sur la page {page_num}: {e}")
                continue

        await browser.close()

    # Dédoublonnage
    df = pd.DataFrame(annonces)
    if not df.empty:
        df = df.drop_duplicates(subset=['titre', 'prix_affiche_fcfa', 'quartier'])
    return df

# Exécution dans Jupyter
df_expat = await scraper_expat_dakar_robuste(nb_pages=50)

# Exportation au format CSV unifié (séparateur ;)
df_expat.to_csv("dataset_expat_dakar_50pages.csv", index=False, sep=';', encoding='utf-8-sig')
print(f"\n✅ Extraction Expat-Dakar réussie ! Total d'annonces uniques : {len(df_expat)}")

🚀 Début du scraping Expat-Dakar sur 50 pages...
🚀 Chargement de la page 1/50...
  └ 10 annonces récupérées sur la page 1
🚀 Chargement de la page 2/50...
  └ 10 annonces récupérées sur la page 2
🚀 Chargement de la page 3/50...
  └ 10 annonces récupérées sur la page 3
🚀 Chargement de la page 4/50...
  └ 10 annonces récupérées sur la page 4
🚀 Chargement de la page 5/50...
  └ 9 annonces récupérées sur la page 5
🚀 Chargement de la page 6/50...
  └ 10 annonces récupérées sur la page 6
🚀 Chargement de la page 7/50...
  └ 10 annonces récupérées sur la page 7
🚀 Chargement de la page 8/50...
  └ 10 annonces récupérées sur la page 8
🚀 Chargement de la page 9/50...
  └ 10 annonces récupérées sur la page 9
🚀 Chargement de la page 10/50...
  └ 10 annonces récupérées sur la page 10
🚀 Chargement de la page 11/50...
  └ 10 annonces récupérées sur la page 11
🚀 Chargement de la page 12/50...
  └ 10 annonces récupérées sur la page 12
🚀 Chargement de la page 13/50...
  └ 10 annonces récupérées sur la page

## FUSION DATASET

In [18]:

# ==========================================
# 1. CHARGEMENT DES 3 DATASETS
# ==========================================
print("📂 Chargement des jeux de données...")
df_coin = pd.read_csv("dataset_coinafrique_10pages.csv", sep=";")
df_fb = pd.read_csv("dataset_facebook_500.csv", sep=";")
df_expat = pd.read_csv("dataset_expat_dakar_50pages.csv", sep=";")

# Nettoyage préventif des retours à la ligne dans Expat-Dakar
df_expat['adresse_complete'] = df_expat['adresse_complete'].apply(lambda x: " ".join(str(x).split()))

# ==========================================
# 2. CONCATÉNATION MULTI-SOURCES
# ==========================================
df_fusion = pd.concat([df_coin, df_fb, df_expat], ignore_index=True)
print(f"📊 Nombre d'annonces brutes accumulées : {len(df_fusion)}")

# ==========================================
# 3. NORMALISATION DE LA GÉOLOCALISATION & NLP
# ==========================================

# A. Nettoyage et harmonisation du nom des quartiers
def nettoyer_quartier(q):
    q_str = str(q).replace('location_on', '').replace('Dakar', '').strip()
    q_str = re.sub(r'^[,\s-]+', '', q_str)
    
    # Mapping d'harmonisation des synonymes/variantes de quartiers
    mapping = {
        'mermoz-sacré coeur': 'Mermoz-Sacré Coeur',
        'mermoz': 'Mermoz-Sacré Coeur',
        'sacré-coeur': 'Mermoz-Sacré Coeur',
        'sacre coeur': 'Mermoz-Sacré Coeur',
        'dakar plateau': 'Dakar Plateau',
        'plateau': 'Dakar Plateau',
        'almadies': 'Almadies',
        'yoff': 'Yoff',
        'ouakam': 'Ouakam',
        'mamelles': 'Mamelles',
        'fann': 'Fann',
        'point e': 'Point E',
        'mariste': 'Hann Maristes',
        'hann maristes': 'Hann Maristes',
        'sicap liberté': 'Sicap Liberté',
        'liberté 6': 'Sicap Liberté',
        'ouest foire': 'Ouest Foire',
        'nord foire': 'Nord Foire',
        'ngor': 'Ngor'
    }
    
    q_lower = q_str.lower()
    for key, val in mapping.items():
        if key in q_lower:
            return val
            
    return q_str.title() if q_str else "Dakar Central"

df_fusion['quartier_clean'] = df_fusion['quartier'].apply(nettoyer_quartier)

# B. Extraction NLP avancée du nombre de pièces
def extraire_nb_pieces(titre):
    titre_str = str(titre).lower()
    # Capture: 3 pièces, 2 chambres, 4 chs, etc.
    match_classique = re.search(r'(\d+)\s*(pièces|piece|pieces|chambres|chambre|chbrs|chs)', titre_str)
    if match_classique:
        return float(match_classique.group(1))
    
    # Capture le format F2, F3, F4, F5
    match_f = re.search(r'\bf(\d+)\b', titre_str)
    if match_f:
        return float(match_f.group(1))
        
    # Studios
    if 'studio' in titre_str or 'chambre' in titre_str:
        return 1.0
        
    return None

df_fusion['nb_pieces_estime'] = df_fusion['titre'].apply(extraire_nb_pieces)

# ==========================================
# 4. GÉOLOCALISATION ET FEATURE ENGINEERING SPATIAL
# ==========================================
COORDS_QUARTIERS = {
    "Yoff": (14.7605, -17.4658),
    "Dakar Plateau": (14.6681, -17.4350),
    "Mermoz-Sacré Coeur": (14.7161, -17.4631),
    "Almadies": (14.7458, -17.5192),
    "Ouakam": (14.7231, -17.4886),
    "Ngor": (14.7553, -17.5169),
    "Mamelles": (14.7333, -17.4983),
    "Fann": (14.6892, -17.4653),
    "Point E": (14.6935, -17.4581),
    "Hann Maristes": (14.7258, -17.4322),
    "Sicap Liberté": (14.7125, -17.4528),
    "Ouest Foire": (14.7431, -17.4619),
    "Nord Foire": (14.7486, -17.4683)
}

PTS_STRATEGIQUES = {
    "Plateau_CBD": (14.6681, -17.4350), # Centre financier/affaires
    "VDN_Axe": (14.7100, -17.4680)      # Axe d'activité VDN
}

def haversine(coord1, coord2):
    lat1, lon1 = coord1
    lat2, lon2 = coord2
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2 +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(dlon / 2) ** 2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def calculer_distances(row):
    q = row['quartier_clean']
    coords = COORDS_QUARTIERS.get(q, (14.7166, -17.4666))
    
    dist_plateau = haversine(coords, PTS_STRATEGIQUES["Plateau_CBD"])
    dist_vdn = haversine(coords, PTS_STRATEGIQUES["VDN_Axe"])
    
    return pd.Series({
        'latitude': coords[0],
        'longitude': coords[1],
        'dist_centre_plateau_km': round(dist_plateau, 2),
        'dist_vdn_km': round(dist_vdn, 2)
    })

df_geo = df_fusion.apply(calculer_distances, axis=1)
df_fusion = pd.concat([df_fusion, df_geo], axis=1)

# ==========================================
# 5. DÉDOUBLONNAGE MULTI-SOURCES STRATÉGIQUE
# ==========================================

# A. Filtrage des prix aberrants / ventes (garder les loyers entre 20 000 FCFA et 5 000 000 FCFA)
df_clean = df_fusion[(df_fusion['prix_affiche_fcfa'] >= 20000) & (df_fusion['prix_affiche_fcfa'] <= 5000000)].copy()

# B. Création de la clé unique pour détecter les doublons inter-plateformes
df_clean['cle_doublon'] = (
    df_clean['prix_affiche_fcfa'].astype(str) + "_" +
    df_clean['quartier_clean'] + "_" +
    df_clean['nb_pieces_estime'].fillna(0).astype(str)
)

# C. Suppression des doublons (en conservant la première apparition)
nb_avant = len(df_clean)
df_final = df_clean.drop_duplicates(subset=['cle_doublon'], keep='first').copy()
nb_apres = len(df_final)

print(f"🧹 Dédoublonnage terminé : {nb_avant - nb_apres} doublons/redondances supprimés.")
print(f"✨ Taille finale du dataset nettoyé & unique : {nb_apres} annonces.")

# ==========================================
# 6. EXPORTATION FINAL COMPATIBLE EXCEL
# ==========================================
df_final.to_csv("dataset_dakar_master_multisources.csv", index=False, sep=';', encoding='utf-8-sig')
print("\n✅ Fichier 'dataset_dakar_master_multisources.csv' généré avec succès !")

📂 Chargement des jeux de données...
📊 Nombre d'annonces brutes accumulées : 1695
🧹 Dédoublonnage terminé : 576 doublons/redondances supprimés.
✨ Taille finale du dataset nettoyé & unique : 1017 annonces.

✅ Fichier 'dataset_dakar_master_multisources.csv' généré avec succès !


In [4]:
# extrait du dataset final pour vérification rapide
print("\n📊 Aperçu des 15 premières annonces du dataset final :")
readcsv = pd.read_csv("dataset_dakar_master_multisources.csv", sep=';')
print(readcsv.head(15))


📊 Aperçu des 15 premières annonces du dataset final :
                                              titre  prix_affiche_fcfa  \
0            Location appartement 3 pièces à Mermoz             350000   
1            Location appartement rdc  aux Almadies             800000   
2               Location  appartement  aux Almadies            3000000   
3                  Location appartement  au mamelle             750000   
4   location appartements 03 pièces - sicap liberté             375000   
5                 Location appartement f5 à Mamelle             880000   
6       Location appartement 4 pièces  aux almadies            1880000   
7        Location mini-studio  - Cité adcna mamelle             150000   
8           Location appartement 3 pièces à Mamelle             150000   
9           Location  appartement 2 pièces à Mermoz             350000   
10    Location appartement  4 pièces  - Cité biagui             500000   
11  Location appartement 3 pièces  à sacre-coeur 03      